# V31: fresh Qwen3 4B versus 8B

One 90-minute offline two-T4 pilot. Both arms use the same 1,024 training
pairs and 512 development pairs, with 32 **accepted** updates each. The initial
128-update draft was stopped before GPU on a historical-runtime planning
screen; its source and expanded inputs remain preserved. This is an early
learning and throughput comparison, not a full-training or official-score forecast.

Each joint prompt/answer-A/answer-B input produces three logits through a new
classification head. NF4 base weights stay frozen while QLoRA adapters and the
head learn the original human votes. Training swaps answer order; evaluation
averages original and aligned swapped probabilities. Development groups are
excluded from these arms' fitting, but were historically inspected. Base-model
pretraining exposure is unknown. Architecture/pretraining and numerical batch
recipes also differ, so parameter count alone is not isolated.

[Protocol](../docs/SPAN_031_PLAN.md) ·
[Pre-GPU exposure correction](../docs/SPAN_031_EXPOSURE_REVISION.md).
The leaderboard objective still requires an actual official rank of 19 or better.

In [1]:
from pathlib import Path
import json,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(REPO/'src'))
from arena.data import ROOT,CLASSES
from arena.span_heads import metrics
PUB=REPO/'results/span_031';RUN=ROOT/'runs/span_031'
for name in ['protocol_revision.json','pilot_preparation.json','size_pilot_cell_rehearsals.json']:
 p=PUB/name
 if p.exists():
  value=json.loads(p.read_text());print(name,{k:v for k,v in value.items() if k not in ['rehearsals'] and 'sha256' not in k})
reconciliation=PUB/'save_reconciliation.json'
if reconciliation.exists():print('Original embedded save reconciliation:',json.loads(reconciliation.read_text()))
role='size_pilot_compact' if (PUB/'jobs/size_pilot_compact').exists() else 'size_pilot'
for name in ['push.json','source_verification.json','controller_state.json','progress.json']:
 p=PUB/'jobs'/role/name
 if p.exists():
  value=json.loads(p.read_text());print(name,value.get('latest',value))
result=PUB/'H02_results.json'
if not result.exists():
 print('No audited quality result yet. Availability and CPU rehearsals do not prove GPU fit or quality.')
else:
 r=json.loads(result.read_text());print('Actual decision:',r['decision'])
 print('Exposure-matched comparison valid:',r['quality_comparison_valid'],'; longer training justified:',r['longer_training_justified'])
 print('Raw paired prompt-group gain/95% interval:',r.get('raw_paired_gain'))
 print('Temperature CV is secondary:',r.get('temperature_cv_diagnostic'))
 print('Actual public GPU charge:',r['charged_gpu_seconds'],'two-T4 session-seconds / 5400')
 rows=[];classes=[]
 for name,a in r['arms'].items():
  if not a.get('complete'):
   print(name,a);continue
  m=a['raw_metrics']
  rows.append(dict(model=name,**{k:m[k] for k in ['log_loss','accuracy','balanced_accuracy','macro_f1','brier']},
   accepted_updates=a['accepted_updates'],accepted_rows=a['global_accepted_rows'],unique_rows=a['global_accepted_unique_rows'],
   accepted_tokens=a['global_accepted_tokens'],train_seconds=a['training_seconds'],
   both_order_prediction_seconds=a['prediction_seconds'],runtime_screen_seconds=a['projection_seconds'],
   peak_memory_fraction=max(x['hardware']['peak_memory_fraction'] for x in a['per_rank'])))
  for i,label in enumerate(CLASSES):
   classes.append(dict(model=name,label=label,loss=m['class_loss'][i],recall=m['class_recall'][i],mean_probability=m['mean_class_probability'][i]))
  print(name,'order diagnostics:',a['order_diagnostics'])
 if rows:display(pd.DataFrame(rows));display(pd.DataFrame(classes))
 arrays=RUN/'size_comparison.npz'
 if arrays.exists():
  saved=np.load(arrays);ids=json.loads((RUN/'pilot_ids.json').read_text())
  train=pd.read_csv(RUN/'pilot_rows.csv.gz').set_index('id').loc[ids['train']]
  counts=train[CLASSES].sum().to_numpy(float);prior=counts/counts.sum()
  hard=np.zeros(3);hard[prior.argmax()]=1
  base=[]
  for name,p in [('training_class_prior',prior),('most_frequent_class_hard',hard)]:
   m=metrics(saved['labels'],np.tile(p,(len(saved['labels']),1)))
   base.append(dict(model=name,**{k:m[k] for k in ['log_loss','accuracy','macro_f1','brier']}))
  display(pd.DataFrame(base))
  print('Baselines use training labels only. Hard-majority log loss uses the metric function probability floor 1e-12.')
 complete={name:a for name,a in r['arms'].items() if a.get('complete')}
 if complete:
  fig,axes=plt.subplots(1,2,figsize=(10,3.5))
  for name,a in complete.items():
   curve=pd.DataFrame(a['training_curve'])
   if len(curve):axes[0].plot(curve.elapsed_seconds/60,curve.accepted_updates,label=name)
   loss=pd.DataFrame(a['loss_curve'])
   if len(loss) and 'loss' in loss:axes[1].plot(loss.step,loss.loss,label=name)
  axes[0].set(xlabel='Elapsed training minutes',ylabel='Actual accepted updates',title='Measured throughput')
  axes[1].set(xlabel='Nominal optimizer cycle',ylabel='Reported training loss',title='Training telemetry')
  for ax in axes:ax.legend()
  fig.tight_layout();(RUN/'figures').mkdir(exist_ok=True)
  fig.savefig(RUN/'figures/size_training.png',dpi=160,bbox_inches='tight');plt.show()
 print('Limitations:',r['comparison_limitations'])
 print('This span performs no official submission; no leaderboard improvement is established.')

protocol_revision.json {'at': '2026-11-09T06:08:46.553466+00:00', 'revision': 2, 'reason': 'Pre-GPU runtime risk in unexecuted 128-update draft; preserve 5400-second allocation', 'historical_training_seconds': 5913.4362790000005, 'historical_nominal_cycles': 256, 'reference_seconds_per_cycle': 23.099360464843752, 'projected_4b_128_training_seconds': 2956.7181395000002, 'initial_4b_training_window_seconds': 1980, 'historical_summary_path': '/home/tarstars/database/llm_classification_finetuning/runs/span_007/jobs/continuation/output/continuation/summary.json', 'causal_throughput_benchmark': False, 'larger_model_throughput_measured': False, 'new_quality_observations': 0, 'new_gpu_executions': 0, 'original_deadline_unchanged': True, 'quota_anchor_unchanged': True, 'quota_start_unchanged': True, 'public_cap_unchanged': True}
pilot_preparation.json {'at': '2026-11-09T06:17:34.655870+00:00', 'prepared': True, 'protocol_revision': 2, 'sample_rows': {'train': 1024, 'dev': 512}, 'compressed_csv_